In [5]:
pip install PyPortfolioOpt

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import warnings; warnings.filterwarnings("ignore")
import os, pickle, logging
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import statsmodels.api as sm
from pypfopt import EfficientFrontier, expected_returns, risk_models, BlackLittermanModel
from pypfopt.risk_models import CovarianceShrinkage

logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)

# Data input
DATA_PATH        = "/content/drive/MyDrive/Klurvss3am/from 2020_all_sp500_tickers_since_2020.csv"
OUT              = "/content/drive/MyDrive/Klurvss3am/latest"
os.makedirs(OUT, exist_ok=True)

RISK_FREE_ANNUAL = 0.0525          # US risk-free rate
RISK_FREE_DAILY  = RISK_FREE_ANNUAL / 252
DATE_START       = None  # use full dataset
DATE_END         = None  # use full dataset

# Rebalancing frequencies mapped to pandas offsets
REBAL_FREQS = {
    "Weekly"    : "W-FRI",
    "Monthly"   : "BME",
    "Quarterly" : "BQS",
    "Yearly"    : "BYS",
    "Buy & Hold": None,
}

OPTIMISERS = ["Max Sharpe", "Min Variance", "Black-Litterman", "Risk Parity"]

TRANSACTION_COST_BPS = 5       # 5 bps per side = 0.05% per trade
TRANSACTION_COST     = TRANSACTION_COST_BPS / 10_000

# Dynamic risk budgeting — cap any single ticker's weight at this vol threshold
DYNAMIC_RISK_BUDGET       = True
VOL_WEIGHT_CAP            = 0.30
ROLLING_VOL_WINDOW        = 21

# HELPERS

def max_drawdown(cum_series):
    return (cum_series / cum_series.cummax() - 1).min()

def annualised_return(ret_series):
    return (1 + ret_series).prod() ** (252 / len(ret_series)) - 1

def annualised_vol(ret_series):
    return ret_series.std() * np.sqrt(252)

def sharpe(ret_series):
    ar = annualised_return(ret_series)
    av = annualised_vol(ret_series)
    return (ar - RISK_FREE_ANNUAL) / av if av > 0 else np.nan

def calmar(ret_series):
    cum = (1 + ret_series).cumprod()
    mdd = max_drawdown(cum)
    ar  = annualised_return(ret_series)
    return ar / abs(mdd) if mdd != 0 else np.nan

def portfolio_metrics(ret_series, label, bench_log_ret=None):
    cum   = (1 + ret_series).cumprod()
    ar    = annualised_return(ret_series)
    av    = annualised_vol(ret_series)
    sr    = sharpe(ret_series)
    mdd   = max_drawdown(cum)
    cal   = calmar(ret_series)
    cum_r = cum.iloc[-1] - 1

    row = {"Label": label, "Ann Return": ar, "Ann Vol": av,
           "Sharpe": sr, "Max Drawdown": mdd, "Calmar": cal,
           "Cumulative Return": cum_r}

    if bench_log_ret is not None:
        aligned = pd.concat([ret_series, bench_log_ret], axis=1).dropna()
        aligned.columns = ["p", "b"]
        if len(aligned) > 2 and aligned["b"].var() > 0:
            beta  = np.cov(aligned["p"], aligned["b"])[0, 1] / aligned["b"].var()
            capm  = RISK_FREE_ANNUAL + beta * (aligned["b"].mean() * 252 - RISK_FREE_ANNUAL)
            alpha = ar - capm
            row["Beta"]  = beta
            row["Alpha"] = alpha
    return row

# STAGE 1 — DATA LOADING
print("\n" + "="*70)
print("STAGE 1 — DATA LOADING & CLEANING")
print("="*70)

raw      = pd.read_csv(DATA_PATH, index_col="Date", parse_dates=True)
raw.sort_index(inplace=True)
# CSV already capped to 2020-2026
pivot_raw = raw.copy()
dead = pivot_raw.columns[pivot_raw.isna().all()].tolist()
flat = pivot_raw.columns[pivot_raw.nunique() <= 1].tolist()
pivot_raw.drop(columns=dead + flat, inplace=True)

print(f"  ✅ Loaded {len(pivot_raw):,} rows × {pivot_raw.shape[1]} columns")
print(f"  Date range: {pivot_raw.index.min()} → {pivot_raw.index.max()}")





STAGE 1 — DATA LOADING & CLEANING
  ✅ Loaded 1,601 rows × 503 columns
  Date range: 2020-01-02 00:00:00 → 2026-05-15 00:00:00


In [9]:
print("Head of the dataset:")
display(pivot_raw.head(5))

print("Tail of the dataset:")
display(pivot_raw.tail(5))

Head of the dataset:


,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,82.210464,72.333893,68.696259,NaN,77.328728,41.268997,190.896103,334.429993,107.791672,38.509460,...,23.394993,138.114822,51.483532,53.306408,74.100807,63.830002,90.903389,137.576370,259.140015,126.422287
2020-01-03,80.890503,71.630646,68.044182,NaN,76.386009,41.221451,190.578232,331.809998,105.894150,38.434303,...,23.434790,136.066208,51.731140,52.877850,74.536957,63.000000,90.618675,137.216797,256.049988,126.441147
2020-01-06,81.129631,72.201416,68.581184,NaN,76.786232,41.383106,189.333694,333.709991,104.649994,38.133636,...,23.387030,135.796890,51.656849,53.283848,74.054405,62.570000,90.565315,136.423782,258.010010,125.470390
2020-01-07,81.378311,71.861839,68.189934,NaN,76.359352,41.040779,185.246017,333.390015,107.030876,37.674290,...,23.235786,136.441315,51.549561,52.847775,73.776001,64.589996,90.725426,136.303909,256.470001,125.894493
2020-01-08,82.181755,73.017830,68.673256,NaN,76.670609,40.631893,185.609406,337.869995,107.997513,37.256714,...,23.323345,137.297318,51.500042,52.050827,74.026558,67.599998,90.885597,137.936005,247.639999,125.621162


Tail of the dataset:


,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2026-05-11,111.459999,292.679993,202.779999,137.050003,82.559998,93.830002,172.350006,246.149994,422.730011,79.839996,...,23.420000,99.370003,80.599998,149.679993,112.000000,73.169998,150.289993,79.580002,216.960007,76.669998
2026-05-12,112.900002,294.799988,207.860001,135.479996,84.349998,94.309998,169.770004,240.830002,419.649994,80.730003,...,23.450001,97.250000,79.900002,150.630005,111.540001,72.120003,152.860001,83.370003,241.789993,76.940002
2026-05-13,112.739998,298.869995,208.500000,132.970001,83.830002,93.320000,159.639999,236.070007,432.390015,82.889999,...,23.100000,96.209999,79.910004,151.570007,109.010002,69.779999,149.770004,82.720001,246.759995,74.339996
2026-05-14,113.260002,298.209991,210.770004,133.669998,84.900002,93.459999,163.990005,237.009995,426.790009,81.410004,...,23.350000,95.430000,80.029999,152.779999,109.440002,71.529999,150.630005,82.650002,258.100006,75.480003
2026-05-15,111.699997,300.230011,210.389999,132.850006,84.470001,93.980003,168.820007,247.600006,417.489990,80.400002,...,22.680000,95.419998,77.919998,157.919998,108.120003,70.360001,149.970001,83.699997,259.350006,74.220001


In [9]:

# STAGE 2 — LOG RETURNS & DESCRIPTIVE STATS
print("\n" + "="*70)
print("STAGE 2 — LOG RETURNS & FILTERING")
print("="*70)

# Compute log returns on rawdata to correctly identify tickers with insufficient history
log_raw_unfilled = np.log(pivot_raw / pivot_raw.shift(1)).dropna(how="all")
notna_pct        = log_raw_unfilled.notna().mean()
valid            = notna_pct[notna_pct > 0.95].index.tolist()
print(f"  ✅ Retained {len(valid)} tickers with ≥95% data")

# Now fill pivot for price-based use and return computation
pivot_df = pivot_raw[valid].copy().ffill().bfill()
log_raw  = np.log(pivot_df / pivot_df.shift(1)).dropna(how="all")
log_returns_df = log_raw.copy()
flat2    = log_returns_df.columns[log_returns_df.std() < 1e-8].tolist()
if flat2:
    print(f"  ⚠️ Dropping {len(flat2)} constant tickers")
    log_returns_df.drop(columns=flat2, inplace=True)
    pivot_df.drop(columns=flat2, inplace=True)
else:
    print(f"  ✅ No constant tickers found")

desc_stats = pd.DataFrame({
    "Mean": log_returns_df.mean(), "Std Dev": log_returns_df.std(),
    "Skewness": log_returns_df.skew(), "Kurtosis": log_returns_df.kurtosis(),
}).sort_index()
desc_stats.to_csv(f"{OUT}/descriptive_stats.csv")

benchmark_log = log_returns_df.mean(axis=1)
benchmark_cum = (1 + benchmark_log).cumprod()

# Benchmark cumulative wealth + rolling Sharpe
_rolling_sr = (
    benchmark_log.rolling(252).mean() * 252
    / (benchmark_log.rolling(252).std() * np.sqrt(252))
)
fig, _axes35 = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                             gridspec_kw={"height_ratios": [2, 1], "hspace": 0.06})
_ax1 = _axes35[0]
_ax1.plot(benchmark_cum.index, benchmark_cum.values,
          color="#263238", linewidth=1.8, label="Equal-weighted benchmark")
_ax1.fill_between(benchmark_cum.index, 1, benchmark_cum.values, alpha=0.10, color="#263238")
_ax1.axhline(1, color="#bbb", linewidth=0.8, linestyle=":")
_ax1.set_ylabel("Growth of $1", fontsize=10)
_ax1.set_title(
    "Figure 3.5: Equal-Weighted S&P 500 Benchmark — Cumulative Wealth and Rolling Sharpe Ratio",
    fontsize=11, fontweight="bold"
)
_ax1.legend(fontsize=9)
_ax1.grid(True, linestyle="--", alpha=0.4)

_ax2 = _axes35[1]
_ax2.plot(_rolling_sr.index, _rolling_sr.values, color="#1565C0", linewidth=1.4,
          label="1-year rolling Sharpe")
_ax2.axhline(
    (annualised_return(benchmark_log) - RISK_FREE_ANNUAL) / annualised_vol(benchmark_log),
    color="#E53935", linewidth=1.2, linestyle="--", label="Full-sample Sharpe"
)
_ax2.axhline(0, color="#999", linewidth=0.7)
_ax2.fill_between(_rolling_sr.index, 0, _rolling_sr.values,
                  where=_rolling_sr > 0, alpha=0.15, color="#1565C0")
_ax2.fill_between(_rolling_sr.index, 0, _rolling_sr.values,
                  where=_rolling_sr < 0, alpha=0.15, color="#E53935")
_ax2.set_ylabel("Rolling Sharpe", fontsize=10)
_ax2.set_xlabel("Date", fontsize=10)
_ax2.legend(fontsize=8)
_ax2.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(f"{OUT}/fig35_benchmark_wealth_rolling_sharpe.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig35_benchmark_wealth_rolling_sharpe.png")

print(f"  Final cleaned price matrix shape: {pivot_df.shape}")
print(f"  Log returns shape: {log_returns_df.shape}")

# STAGE 3 — CORRELATION CLUSTERING
print("\n" + "="*70)
print("STAGE 3 — CORRELATION CLUSTERING HEATMAP")
print("="*70)

corr_matrix = log_returns_df.corr().replace([np.inf, -np.inf], np.nan).fillna(0)
dist_arr    = (1 - corr_matrix).values.copy()
np.fill_diagonal(dist_arr, 0)
condensed   = squareform(dist_arr)
row_link    = linkage(condensed, method="ward")

sample = log_returns_df.columns[:50].tolist()
cg = sns.clustermap(corr_matrix.loc[sample, sample], cmap="coolwarm",
    figsize=(18, 18), annot=False, linewidths=0.3,
    cbar_kws={"label": "Correlation", "shrink": 0.7})
plt.setp(cg.ax_heatmap.yaxis.get_majorticklabels(), rotation=0,  fontsize=5)
plt.setp(cg.ax_heatmap.xaxis.get_majorticklabels(), rotation=90, fontsize=5)
plt.suptitle("S&P 500 Clustered Correlation Heatmap", fontsize=14, y=1.01)
plt.savefig(f"{OUT}/correlation_clustermap.png", dpi=200, bbox_inches="tight")
plt.close()
print("  Saved: correlation_clustermap.png")


STAGE 2 — LOG RETURNS & FILTERING
  ✅ Retained 488 tickers with ≥95% data
  ✅ No constant tickers found
  Saved: fig35_benchmark_wealth_rolling_sharpe.png
  Final cleaned price matrix shape: (1601, 488)
  Log returns shape: (1600, 488)

STAGE 3 — CORRELATION CLUSTERING HEATMAP
  Saved: correlation_clustermap.png


In [10]:

# STAGE 4 — PCA
print("\n" + "="*70)
print("STAGE 4 — PCA")
print("="*70)

scaler    = StandardScaler()
X_tickers = scaler.fit_transform(log_returns_df.values).T  # shape: (n_tickers, n_days)

n_tickers = X_tickers.shape[0]
n_days    = X_tickers.shape[1]

# Full PCA — all components so we can plot the complete cumulative curve
pca_full  = PCA(n_components=min(n_tickers, n_days)).fit(X_tickers)
ev        = pca_full.explained_variance_ratio_
cum_ev    = np.cumsum(ev)

# How many PCs needed to reach 90%?
n_90pct   = int(np.searchsorted(cum_ev, 0.90)) + 1
print(f"  PCs needed for 90% variance: {n_90pct}")
print(f"  Cumulative variance (PC1-10): {cum_ev[9]:.2%}")

#Scree plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: first 50 component
ax = axes[0]
ax.bar(range(1, 51), ev[:50], alpha=0.7, color="steelblue", label="Individual")
ax.step(range(1, 51), cum_ev[:50], color="red", linewidth=1.8, label="Cumulative")
ax.axhline(0.9, color="grey", linestyle="--", linewidth=1.2, label="90% Threshold")
ax.set_xlabel("Number of Principal Components")
ax.set_ylabel("Explained Variance Ratio")
ax.set_title("Scree Plot — First 50 PCs")
ax.legend(); ax.grid(True, linestyle="--", alpha=0.4)

# Right: full cumulative curve across ALL components
ax2 = axes[1]
ax2.plot(range(1, len(cum_ev)+1), cum_ev,
         color="blue", linewidth=1.5, marker=".", markersize=2)
ax2.axhline(0.90, color="red", linestyle="--", linewidth=1.2, label="90% Threshold")
ax2.axvline(n_90pct, color="orange", linestyle=":", linewidth=1.2,
            label=f"{n_90pct} PCs → 90%")
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Variance")
ax2.set_title("PCA — Cumulative Explained Variance (S&P 500)")
ax2.legend(fontsize=9); ax2.grid(True)

plt.suptitle("PCA on S&P 500 Log Returns (488 tickers × 1,600 days)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT}/pca_scree_plot.png", dpi=150, bbox_inches="tight")
plt.close()

# ─2D Projection
n_comp   = min(45, n_tickers - 1)
pca_proj = PCA(n_components=n_comp).fit_transform(X_tickers)
pca_df   = pd.DataFrame(pca_proj, columns=[f"PC{i+1}" for i in range(n_comp)])
pca_df["Ticker"] = log_returns_df.columns

# Scatter: tickers coloured by sector-proxy
fig, ax = plt.subplots(figsize=(18, 12))
ax.scatter(pca_df["PC1"], pca_df["PC2"],
           alpha=0.65, s=35, color="steelblue", edgecolors="none")
for _, row in pca_df.iterrows():
    ax.text(row["PC1"] + 0.15, row["PC2"] + 0.15,
            row["Ticker"], fontsize=4.5, alpha=0.85)
ax.set_title("PCA Projection of S&P 500 Tickers (based on Log Returns)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Principal Component 1", fontsize=11)
ax.set_ylabel("Principal Component 2", fontsize=11)
ax.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/pca_2d_projection.png", dpi=180, bbox_inches="tight")
plt.close()
print("  Saved: pca_scree_plot.png, pca_2d_projection.png")




STAGE 4 — PCA
  PCs needed for 90% variance: 249
  Cumulative variance (PC1-10): 32.62%
  Saved: pca_scree_plot.png, pca_2d_projection.png


In [11]:

# STAGE 5 — STRATEGY SELECTION

print("\n" + "="*70)
print("STAGE 5 — STRATEGY SELECTION")
print("="*70)

volatility    = log_returns_df.std()
mean_ret      = log_returns_df.mean()
sharpe_ratios = (mean_ret - RISK_FREE_DAILY) / volatility

top_sharpe  = sharpe_ratios.sort_values(ascending=False).head(10).index.tolist()
low_vol     = volatility.sort_values().head(10).index.tolist()

kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
pca_df["Cluster"] = kmeans.fit_predict(pca_df[["PC1","PC2"]])
pca_cluster = []
for c in range(8):
    cd  = pca_df[pca_df["Cluster"] == c]
    if len(cd) == 0:
        continue
    ctr = kmeans.cluster_centers_[c]
    idx = cd[["PC1","PC2"]].sub(ctr).pow(2).sum(1).idxmin()
    pca_cluster.append(pca_df.loc[idx, "Ticker"])

# Use 10-ticker universes for richer optimisation
strategies = {
    "High Sharpe"   : top_sharpe,
    "Low Volatility": low_vol,
    "PCA Cluster"   : pca_cluster,
}

print(f"★ Top 10 High Sharpe Tickers: {top_sharpe}")
print(f"★ Top 10 Low Volatility Tickers: {low_vol}")
print(f"★ PCA Cluster Representatives: {pca_cluster}")

# ── Fig 5.4 — §5.1  Universe profile comparison (5 metrics side-by-side) ─────
_univ_names   = ["High Sharpe", "Low Volatility", "PCA Cluster"]
_univ_tickers = [top_sharpe, low_vol, pca_cluster]
_univ_colors  = ["#1565C0", "#6A1B9A", "#2E7D32"]

_u_ret, _u_vol, _u_sr, _u_corr, _u_beta = [], [], [], [], []
for _tks in _univ_tickers:
    _valid_tks = [t for t in _tks if t in log_returns_df.columns]
    _ew_ret = log_returns_df[_valid_tks].mean(axis=1)
    _u_ret.append(annualised_return(_ew_ret))
    _u_vol.append(annualised_vol(_ew_ret))
    _u_sr.append(sharpe(_ew_ret))
    _corr_sub = log_returns_df[_valid_tks].corr()
    _mask = np.triu(np.ones(_corr_sub.shape), k=1).astype(bool)
    _u_corr.append(float(_corr_sub.where(_mask).stack().mean()))
    _aligned = pd.concat([_ew_ret, benchmark_log], axis=1).dropna()
    _aligned.columns = ["p", "b"]
    _beta_val = (np.cov(_aligned["p"], _aligned["b"])[0, 1] / _aligned["b"].var()
                 if _aligned["b"].var() > 0 else np.nan)
    _u_beta.append(_beta_val)

_bench_ar  = annualised_return(benchmark_log)
_bench_vol = annualised_vol(benchmark_log)
_bench_sr  = sharpe(benchmark_log)

_metrics_cfg = [
    ("Ann. Return",        _u_ret,  True,  _bench_ar),
    ("Ann. Volatility",    _u_vol,  True,  _bench_vol),
    ("Sharpe Ratio",       _u_sr,   False, _bench_sr),
    ("Avg Pairwise Corr.", _u_corr, True,  None),
    ("Beta vs EW Bench",   _u_beta, False, 1.0),
]

fig, _axes54 = plt.subplots(1, 5, figsize=(16, 5))
for _ax, (_title, _vals, _is_pct, _bv) in zip(_axes54, _metrics_cfg):
    _bars = _ax.bar(_univ_names, _vals, color=_univ_colors, width=0.55,
                    edgecolor="white", linewidth=0.8)
    if _bv is not None:
        _ax.axhline(_bv, color="#E53935", linewidth=1.2, linestyle="--")
    _ax.set_title(_title, fontsize=9.5, fontweight="bold")
    _ax.set_xticks(range(3))
    _ax.set_xticklabels(["HS", "LV", "PCA"], fontsize=8.5)
    if _is_pct:
        _ax.yaxis.set_major_formatter(
            plt.FuncFormatter(lambda x, _p: f"{x:.0%}"))
    for _bar, _v in zip(_bars, _vals):
        _lbl = f"{_v:.1%}" if _is_pct else f"{_v:.2f}"
        _ax.text(_bar.get_x() + _bar.get_width() / 2,
                 _bar.get_height() + abs(_bar.get_height()) * 0.02,
                 _lbl, ha="center", va="bottom", fontsize=7.5, fontweight="bold")
    _ax.grid(True, linestyle="--", alpha=0.35)

fig.suptitle(
    "Figure 5.4: Key Statistics — Equal-Weighted Universe Comparison, 2020–2026\n"
    "HS = High Sharpe  |  LV = Low Volatility  |  PCA = PCA Cluster  "
    "(dashed red = EW S&P 500 benchmark where applicable)",
    fontsize=10, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(f"{OUT}/fig54_universe_profiles.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig54_universe_profiles.png")

# ── Fig 5.6 — §5.1  Three-universe equal-weighted cumulative wealth curves ────
fig, _ax56 = plt.subplots(figsize=(13, 6))
_uc_map = {
    "High Sharpe":    ("#1565C0", top_sharpe),
    "Low Volatility": ("#6A1B9A", low_vol),
    "PCA Cluster":    ("#2E7D32", pca_cluster),
}
for _uname, (_col, _tks) in _uc_map.items():
    _valid_tks = [t for t in _tks if t in log_returns_df.columns]
    _ew_cum = (1 + log_returns_df[_valid_tks].mean(axis=1)).cumprod()
    _sr_val = sharpe(log_returns_df[_valid_tks].mean(axis=1))
    _ax56.plot(_ew_cum.index, _ew_cum / _ew_cum.iloc[0],
               color=_col, linewidth=2.0,
               label=f"{_uname} (EW, SR={_sr_val:.2f})")

_bench_norm56 = benchmark_cum / benchmark_cum.iloc[0]
_ax56.plot(_bench_norm56.index, _bench_norm56.values,
           color="#263238", linewidth=1.6, linestyle=":",
           label=f"Market EW Benchmark (SR={_bench_sr:.2f})")
_ax56.axhline(1, color="#ccc", linewidth=0.7)
_ax56.set_ylabel("Growth of $1", fontsize=10)
_ax56.set_xlabel("Date", fontsize=10)
_ax56.set_title(
    "Figure 5.6: Equal-Weighted Cumulative Wealth by Universe, 2020–2026\n"
    "(No optimisation applied — pure universe comparison)",
    fontsize=11, fontweight="bold"
)
_ax56.legend(fontsize=9)
_ax56.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(f"{OUT}/fig56_universe_wealth_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig56_universe_wealth_curves.png")




STAGE 5 — STRATEGY UNIVERSE SELECTION
★ Top 10 High Sharpe Tickers: ['FIX', 'PWR', 'NVDA', 'STX', 'AVGO', 'EME', 'LLY', 'VRT', 'MCK', 'HWM']
★ Top 10 Low Volatility Tickers: ['JNJ', 'KO', 'PG', 'CL', 'BRK-B', 'RSG', 'VZ', 'WM', 'KMB', 'PEP']
★ PCA Cluster Representatives: ['OXY', 'V', 'JNJ', 'TSLA', 'TMUS', 'TPR', 'UDR', 'POOL']
  Saved: fig54_universe_profiles.png
  Saved: fig56_universe_wealth_curves.png


In [12]:

# ML FORECAST ACCURACY COMPARISON
print("\n" + "="*70)
print("STAGE 6 — ML FORECAST ACCURACY COMPARISON")
print("="*70)

# Use all strategy tickers, train on first 70%, test on last 30%
forecast_tickers = list(dict.fromkeys(top_sharpe + low_vol + pca_cluster))
LOOKBACK  = 60
TEST_FRAC = 0.30

def make_supervised(series, lookback):
    """Convert a 1-D series into (X, y) sliding windows."""
    arr = series.values
    X, y = [], []
    for i in range(lookback, len(arr)):
        X.append(arr[i-lookback:i])
        y.append(arr[i])
    return np.array(X), np.array(y)

def directional_accuracy(y_true, y_pred):
    return np.mean(np.sign(y_true) == np.sign(y_pred))

def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

# ── Per-ticker ML evaluation ──────────────────────────────────────────────────
ml_records = []   # one row per (ticker, model)

for ticker in forecast_tickers:
    series = log_returns_df[ticker].dropna()
    if len(series) < LOOKBACK * 2 + 10:
        continue

    X, y = make_supervised(series, LOOKBACK)
    split = int(len(X) * (1 - TEST_FRAC))
    X_tr, X_te = X[:split], X[split:]
    y_tr, y_te = y[:split], y[split:]

    if len(X_te) < 5:
        continue

    models_to_run = {}

    # SVR
    sc = MinMaxScaler()
    Xtr_s = sc.fit_transform(X_tr)
    Xte_s = sc.transform(X_te)
    svr = SVR(kernel="rbf", C=10, epsilon=0.001)
    svr.fit(Xtr_s, y_tr)
    models_to_run["SVR"] = svr.predict(Xte_s)

    # XGBoost
    xgb_m = xgb.XGBRegressor(n_estimators=100, max_depth=4,
                               learning_rate=0.05, random_state=42,
                               verbosity=0)
    xgb_m.fit(X_tr, y_tr)
    models_to_run["XGBoost"] = xgb_m.predict(X_te)

    # Gradient Boosting
    gb = GradientBoostingRegressor(n_estimators=100, max_depth=3,
                                    learning_rate=0.05, random_state=42)
    gb.fit(X_tr, y_tr)
    models_to_run["GradientBoosting"] = gb.predict(X_te)

    # LightGBM
    lgb_m = lgb.LGBMRegressor(n_estimators=100, max_depth=4,
                                learning_rate=0.05, random_state=42,
                                verbose=-1)
    lgb_m.fit(X_tr, y_tr)
    models_to_run["LightGBM"] = lgb_m.predict(X_te)

    # ARIMA (statsmodels) — fit on training log-returns
    try:
        arima_m = sm.tsa.ARIMA(y_tr, order=(2, 0, 2)).fit()
        arima_pred = arima_m.forecast(steps=len(y_te))
        models_to_run["ARIMA"] = np.array(arima_pred)
    except Exception:
        pass

    # Prophet — on price series, predict next N days
    try:
        from prophet import Prophet
        price_series = pivot_df[ticker].dropna()
        n_train = int(len(price_series) * (1 - TEST_FRAC))
        pr_train = price_series.iloc[:n_train].reset_index()
        pr_train.columns = ["ds", "y"]
        pr_train["ds"] = pd.to_datetime(pr_train["ds"])
        pm = Prophet(yearly_seasonality=True, weekly_seasonality=True,
                     daily_seasonality=False)
        pm.fit(pr_train)
        future   = pm.make_future_dataframe(periods=len(y_te), freq="B")
        forecast = pm.predict(future)
        yhat     = forecast["yhat"].values[-len(y_te):]
        # convert price forecast to log-returns
        last_pr  = pr_train["y"].values[-1]
        pred_prices = np.concatenate([[last_pr], yhat])
        prophet_log_ret = np.log(pred_prices[1:] / pred_prices[:-1])
        models_to_run["Prophet"] = prophet_log_ret
    except Exception:
        pass

    # Score each model against y_te
    for model_name, y_pred in models_to_run.items():
        # align lengths
        n = min(len(y_te), len(y_pred))
        yt, yp = y_te[:n], y_pred[:n]
        rmse = np.sqrt(mean_squared_error(yt, yp))
        mae  = mean_absolute_error(yt, yp)
        mp   = mape(yt, yp)
        da   = directional_accuracy(yt, yp)
        ml_records.append({
            "Ticker"              : ticker,
            "Model"               : model_name,
            "RMSE"                : rmse,
            "MAE"                 : mae,
            "MAPE"                : mp,
            "Directional Accuracy": da,
        })

ml_results = pd.DataFrame(ml_records)

# leaderboard
ml_leaderboard = (
    ml_results.groupby("Model")
    .agg(
        Avg_RMSE=("RMSE", "mean"),
        Avg_MAE =("MAE",  "mean"),
        Avg_MAPE=("MAPE", "mean"),
        Avg_Dir_Acc=("Directional Accuracy", "mean"),
        Tickers_Tested=("Ticker", "nunique"),
    )
    .sort_values("Avg_RMSE")
    .reset_index()
)
ml_leaderboard.to_csv(f"{OUT}/ml_model_leaderboard.csv", index=False)
ml_results.to_csv(f"{OUT}/ml_model_detail.csv", index=False)

best_ml_model = ml_leaderboard.iloc[0]["Model"]
print(f"\n  ML Model Leaderboard (ranked by RMSE):")
print(ml_leaderboard.to_string(index=False))
print(f"\n  ★ Best ML model overall: {best_ml_model}")

# chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#2ecc71","#3498db","#e74c3c","#f39c12","#9b59b6","#1abc9c"]

axes[0].barh(ml_leaderboard["Model"], ml_leaderboard["Avg_RMSE"],
             color=colors[:len(ml_leaderboard)])
axes[0].set_title("Average RMSE by Model (lower = better)")
axes[0].set_xlabel("RMSE"); axes[0].invert_xaxis()
axes[0].grid(axis="x", linestyle="--", alpha=0.5)

axes[1].barh(ml_leaderboard["Model"], ml_leaderboard["Avg_Dir_Acc"],
             color=colors[:len(ml_leaderboard)])
axes[1].set_title("Avg Directional Accuracy (higher = better)")
axes[1].set_xlabel("Directional Accuracy")
axes[1].axvline(0.5, color="red", linestyle="--", label="Random (50%)")
axes[1].legend(); axes[1].grid(axis="x", linestyle="--", alpha=0.5)

plt.suptitle("ML Model Forecast Accuracy — S&P 500 Strategy Tickers", fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUT}/ml_model_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: ml_model_comparison.png, ml_model_leaderboard.csv")

# ── Fig 6.6 — §6.1.2  Bias-variance: train vs test RMSE + directional accuracy
# Compute per-ticker train RMSE for each model alongside the existing test RMSE
_train_rmse_records = []
for _ticker in forecast_tickers:
    _series = log_returns_df[_ticker].dropna()
    if len(_series) < LOOKBACK * 2 + 10:
        continue
    _X, _y = make_supervised(_series, LOOKBACK)
    _split = int(len(_X) * (1 - TEST_FRAC))
    _X_tr, _X_te = _X[:_split], _X[_split:]
    _y_tr, _y_te = _y[:_split], _y[_split:]
    if len(_X_te) < 5:
        continue

    _tr_models = {}
    # SVR
    _sc2 = MinMaxScaler()
    _Xtr_s2 = _sc2.fit_transform(_X_tr)
    _svr2 = SVR(kernel="rbf", C=10, epsilon=0.001)
    _svr2.fit(_Xtr_s2, _y_tr)
    _tr_models["SVR"] = _svr2.predict(_sc2.transform(_X_tr))
    # XGBoost
    _xgb2 = xgb.XGBRegressor(n_estimators=100, max_depth=4,
                               learning_rate=0.05, random_state=42, verbosity=0)
    _xgb2.fit(_X_tr, _y_tr)
    _tr_models["XGBoost"] = _xgb2.predict(_X_tr)
    # GradientBoosting
    _gb2 = GradientBoostingRegressor(n_estimators=100, max_depth=3,
                                      learning_rate=0.05, random_state=42)
    _gb2.fit(_X_tr, _y_tr)
    _tr_models["GradientBoosting"] = _gb2.predict(_X_tr)
    # LightGBM
    _lgb2 = lgb.LGBMRegressor(n_estimators=100, max_depth=4,
                                learning_rate=0.05, random_state=42, verbose=-1)
    _lgb2.fit(_X_tr, _y_tr)
    _tr_models["LightGBM"] = _lgb2.predict(_X_tr)
    # ARIMA train residuals
    try:
        _arima2 = sm.tsa.ARIMA(_y_tr, order=(2, 0, 2)).fit()
        _tr_models["ARIMA"] = np.array(_arima2.fittedvalues)
    except Exception:
        pass

    for _mname, _ypred_tr in _tr_models.items():
        _n2 = min(len(_y_tr), len(_ypred_tr))
        _train_rmse_records.append({
            "Ticker": _ticker, "Model": _mname,
            "Train_RMSE": np.sqrt(mean_squared_error(_y_tr[:_n2], _ypred_tr[:_n2]))
        })

_train_rmse_df = (
    pd.DataFrame(_train_rmse_records)
    .groupby("Model")["Train_RMSE"].mean()
    .reset_index()
    .rename(columns={"Train_RMSE": "Avg_Train_RMSE"})
)
_merged66 = ml_leaderboard.merge(_train_rmse_df, on="Model", how="left")
_merged66 = _merged66.sort_values("Avg_RMSE")   # order by test RMSE

fig, _axes66 = plt.subplots(1, 2, figsize=(14, 5.5))
_bar_colors66 = ["#2ecc71","#3498db","#e74c3c","#f39c12","#9b59b6","#1abc9c"]

# Left: train vs test RMSE
_x66 = np.arange(len(_merged66))
_w66 = 0.38
_axes66[0].bar(_x66 - _w66/2, _merged66["Avg_Train_RMSE"], _w66,
               color="#1E88E5", alpha=0.85, label="Training RMSE (in-sample fit)")
_axes66[0].bar(_x66 + _w66/2, _merged66["Avg_RMSE"], _w66,
               color="#E53935", alpha=0.85, label="Test RMSE (walk-forward OOS)")
_axes66[0].set_xticks(_x66)
_axes66[0].set_xticklabels(_merged66["Model"], rotation=15, ha="right", fontsize=9)
_axes66[0].set_ylabel("Average RMSE", fontsize=10)
_axes66[0].set_title("Train vs Test RMSE — Overfitting Gap by Model\n"
                     "Gap = generalisation penalty; ARIMA gap is smallest",
                     fontsize=10, fontweight="bold")
_axes66[0].legend(fontsize=9)
_axes66[0].grid(axis="y", linestyle="--", alpha=0.4)

# Right: directional accuracy
_da_colors66 = ["#1565C0" if d > 0.50 else "#E53935"
                for d in _merged66["Avg_Dir_Acc"]]
_bars66b = _axes66[1].bar(_x66, _merged66["Avg_Dir_Acc"] * 100,
                           color=_da_colors66, edgecolor="white", width=0.6, zorder=3)
_axes66[1].axhline(50, color="#333", linewidth=1.4, linestyle="--",
                   label="Random baseline (50%)")
for _bar66, _da in zip(_bars66b, _merged66["Avg_Dir_Acc"]):
    _axes66[1].text(_bar66.get_x() + _bar66.get_width()/2,
                    _bar66.get_height() + 0.08,
                    f"{_da:.2%}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
_axes66[1].set_xticks(_x66)
_axes66[1].set_xticklabels(_merged66["Model"], rotation=15, ha="right", fontsize=9)
_axes66[1].set_ylabel("Directional Accuracy (%)", fontsize=10)
_axes66[1].set_ylim(48, 56)
_axes66[1].set_title("Directional Accuracy by Model\n"
                     "Blue = above random (50%), red = below",
                     fontsize=10, fontweight="bold")
_axes66[1].legend(fontsize=9)
_axes66[1].grid(axis="y", linestyle="--", alpha=0.4)

fig.suptitle(
    "Figure 6.6: Why Classical Models Win — Bias-Variance Trade-off on Noisy Daily Return Series",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(f"{OUT}/fig66_bias_variance.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig66_bias_variance.png")



STAGE 6 — ML FORECAST ACCURACY COMPARISON

  ML Model Leaderboard (ranked by RMSE):
           Model  Avg_RMSE  Avg_MAE  Avg_MAPE  Avg_Dir_Acc  Tickers_Tested
           ARIMA  0.021148 0.014737  1.164435     0.526856              27
GradientBoosting  0.021566 0.015153  1.579981     0.504089              27
         Prophet  0.021602 0.015066  1.325497     0.524451              27
         XGBoost  0.021610 0.015218  1.722920     0.511624              27
        LightGBM  0.021613 0.015233  1.873169     0.509139              27
             SVR  0.023632 0.017199  3.084904     0.497034              27

  ★ Best ML model overall: ARIMA
  Saved: ml_model_comparison.png, ml_model_leaderboard.csv
  Saved: fig66_bias_variance.png


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# STAGE 7 — REBALANCING BACKTESTER
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("STAGE 7 — REBALANCING BACKTESTER")
print("="*70)

ROLLING_LOOKBACK = 252   # 1-year lookback for each rebalance estimation

def get_weights(prices_window, optimiser):
    """
    Returns a dict of {ticker: weight} for the given price window and optimiser.
    Falls back gracefully to equal-weight if optimisation fails.

    Optimisers:
      Max Sharpe      — maximise Sharpe ratio via mean-variance optimisation
      Min Variance    — minimise portfolio variance regardless of return
      Black-Litterman — market-equilibrium prior blended with 10% return view
      Risk Parity     — equalise each ticker's marginal contribution to variance
    """
    tickers = list(prices_window.columns)
    n       = len(tickers)
    ew      = {t: 1/n for t in tickers}

    try:
        mu  = expected_returns.mean_historical_return(prices_window)
        cov = CovarianceShrinkage(prices_window).ledoit_wolf()

        if optimiser == "Max Sharpe":
            ef = EfficientFrontier(mu, cov)
            ef.max_sharpe()
            return ef.clean_weights()

        elif optimiser == "Min Variance":
            ef = EfficientFrontier(mu, cov)
            ef.min_volatility()
            return ef.clean_weights()

        elif optimiser == "Black-Litterman":
            mkt_w    = np.array([1/n]*n)
            mkt_caps = dict(zip(tickers, mkt_w))
            prior    = 2.5 * cov @ mkt_w
            abs_v    = pd.Series(0.10, index=tickers)
            bl       = BlackLittermanModel(cov, pi=prior,
                                           absolute_views=abs_v,
                                           market_caps=mkt_caps)
            bl_mu    = bl.bl_returns()
            ef       = EfficientFrontier(bl_mu, cov)
            try:
                ef.max_sharpe()
            except Exception:
                ef = EfficientFrontier(bl_mu, cov)
                ef.min_volatility()
            return ef.clean_weights()

        elif optimiser == "Risk Parity":
            # Risk Parity: each ticker contributes equally to total portfolio variance.
            # Solved via scipy minimisation: minimise sum of squared differences
            # between each ticker's marginal risk contribution and the target (1/n).
            from scipy.optimize import minimize

            cov_arr = cov.values

            def risk_contributions(w, cov_arr):
                port_var = w @ cov_arr @ w
                mrc      = cov_arr @ w            # marginal risk contributions
                rc       = w * mrc / port_var     # percentage risk contributions
                return rc

            def risk_parity_objective(w, cov_arr, n):
                rc     = risk_contributions(w, cov_arr)
                target = np.ones(n) / n           # equal risk contribution target
                return np.sum((rc - target) ** 2)

            w0          = np.ones(n) / n          # start from equal weight
            constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
            bounds      = [(0.0, 1.0)] * n        # long-only

            result = minimize(
                risk_parity_objective,
                w0,
                args=(cov_arr, n),
                method="SLSQP",
                bounds=bounds,
                constraints=constraints,
                options={"maxiter": 1000, "ftol": 1e-12},
            )

            if result.success:
                w_rp = result.x
                w_rp = np.clip(w_rp, 0, None)
                w_rp /= w_rp.sum()
                return dict(zip(tickers, w_rp))
            else:
                return ew   # fallback to equal-weight if solver fails

    except Exception:
        return ew

    return ew


def apply_dynamic_risk_budget(weights_dict, prices_window, tickers,
                               vol_cap=VOL_WEIGHT_CAP,
                               vol_window=ROLLING_VOL_WINDOW):
    """
    Post-processing step applied AFTER any optimiser.
    Caps each ticker's weight if its recent volatility is high relative to others.

    Mechanism:
      1. Compute each ticker's rolling annualised vol over vol_window days.
      2. Inverse-vol score: low-vol tickers get higher budget.
      3. Scale the optimiser's weights by inverse-vol scores.
      4. Hard-cap any single weight at vol_cap, then renormalise.

    This is purely additive — it wraps get_weights() output without
    modifying the optimiser itself.
    """
    w_arr = np.array([weights_dict.get(t, 0) for t in tickers])
    if w_arr.sum() == 0:
        return w_arr

    try:
        recent = prices_window.iloc[-vol_window:]
        log_r  = np.log(recent / recent.shift(1)).dropna()
        vol    = log_r.std() * np.sqrt(252)           # annualised vol per ticker
        vol    = vol.reindex(tickers).fillna(vol.mean())

        inv_vol = 1.0 / (vol + 1e-8)                  # inverse-vol scores
        inv_vol /= inv_vol.sum()                       # normalise to sum=1

        # Blend optimiser weights with inverse-vol signal (50/50)
        blended = 0.5 * w_arr + 0.5 * inv_vol.values
        blended = np.clip(blended, 0, vol_cap)         # hard cap
        if blended.sum() > 0:
            blended /= blended.sum()
        return blended

    except Exception:
        return w_arr   # if anything fails, return original weights unchanged


def backtest(prices, log_ret, freq_label, freq_offset, optimiser, bench_log,
             tx_cost=TRANSACTION_COST, use_dynamic_budget=DYNAMIC_RISK_BUDGET):
    """
    Run a full rebalancing backtest with optional transaction costs and
    dynamic risk budgeting.

    Parameters
    ----------
    tx_cost : float
        Fraction deducted per unit of turnover on each rebalance day.
        e.g. 0.0005 = 5 bps per side.  Set to 0 to disable.
    use_dynamic_budget : bool
        If True, post-processes optimiser weights through apply_dynamic_risk_budget()
        before applying them.  Does not change the optimiser itself.
    """
    valid_tickers = [t for t in prices.columns if t in log_ret.columns]
    prices  = prices[valid_tickers].dropna(axis=1, how="any")
    log_ret = log_ret[list(prices.columns)]
    tickers = list(prices.columns)
    n       = len(tickers)

    def _finalise_weights(raw_w_dict, hist_prices):
        """Apply dynamic risk budget then return a normalised weight array."""
        w_arr = np.array([raw_w_dict.get(t, 0) for t in tickers])
        if use_dynamic_budget and len(hist_prices) >= ROLLING_VOL_WINDOW:
            w_arr = apply_dynamic_risk_budget(raw_w_dict, hist_prices, tickers)
        if w_arr.sum() > 0:
            w_arr /= w_arr.sum()
        else:
            w_arr = np.ones(n) / n
        return w_arr

    # ── Buy & Hold: optimise once, then hold forever ──────────────────────────
    if freq_offset is None:
        hist0     = prices.iloc[:ROLLING_LOOKBACK]
        raw_w     = get_weights(hist0, optimiser)
        current_w = _finalise_weights(raw_w, hist0)
        port_ret  = log_ret.dot(current_w)
        cum       = (1 + port_ret).cumprod()
        label     = f"{optimiser} | Buy&Hold"
        m         = portfolio_metrics(port_ret, label, bench_log)
        m["Turnover (ann)"] = 0.0
        m["Tx Cost Drag"]   = 0.0
        return m, cum

    # ── Rebalancing: rebuild weights on each rebalance date ───────────────────
    rebal_dates = pd.date_range(
        start=prices.index[ROLLING_LOOKBACK],
        end=prices.index[-1],
        freq=freq_offset
    )
    rebal_date_set = set(rebal_dates)

    port_ret_list  = []
    current_w      = None
    prev_w         = None
    total_turnover = 0.0   # cumulative one-way turnover across all rebalances
    n_rebalances   = 0

    for date in prices.index:
        is_rebal = (date in rebal_date_set) or (current_w is None)

        if is_rebal:
            hist = prices.loc[prices.index <= date].iloc[-ROLLING_LOOKBACK:]
            if len(hist) >= 20:
                raw_w     = get_weights(hist, optimiser)
                new_w     = _finalise_weights(raw_w, hist)

                # ── Transaction cost: deduct on the rebalance day ─────────────
                if prev_w is not None and tx_cost > 0:
                    turnover = np.sum(np.abs(new_w - prev_w)) / 2  # one-way
                    total_turnover += turnover
                    n_rebalances   += 1
                    # Deduct cost from this day's return
                    if date in log_ret.index:
                        day_r = log_ret.loc[date, tickers].fillna(0).values
                        port_ret_list.append(day_r.dot(new_w) - tx_cost * turnover)
                        current_w = new_w
                        prev_w    = new_w.copy()
                        continue   # already appended — skip the normal append below

                current_w = new_w
                prev_w    = new_w.copy()

        if current_w is not None and date in log_ret.index:
            day_ret = log_ret.loc[date, tickers].fillna(0).values
            port_ret_list.append(day_ret.dot(current_w))

    port_ret = pd.Series(port_ret_list, index=log_ret.index[:len(port_ret_list)])
    cum      = (1 + port_ret).cumprod()
    label    = f"{optimiser} | {freq_label}"

    m = portfolio_metrics(port_ret, label, bench_log)

    # Annualised turnover and cost drag
    years = len(port_ret) / 252
    m["Turnover (ann)"] = (total_turnover / years) if years > 0 else 0.0
    m["Tx Cost Drag"]   = (total_turnover * tx_cost * 2 / years) if years > 0 else 0.0

    return m, cum


# Run all combinations: 5 frequencies × 4 optimisers × 3 strategy universes
all_results  = []          # metric rows
all_cumrets  = {}          # {label: cum_series}

total_combos = len(strategies) * len(OPTIMISERS) * len(REBAL_FREQS)
done = 0

for strat_name, tickers in strategies.items():
    valid    = [t for t in tickers if t in pivot_df.columns and t in log_returns_df.columns]
    prices   = pivot_df[valid].dropna(axis=1, how="any")
    log_ret  = log_returns_df[[t for t in valid if t in log_returns_df.columns]]

    for opt in OPTIMISERS:
        for freq_label, freq_offset in REBAL_FREQS.items():
            done += 1
            label = f"{strat_name} | {opt} | {freq_label}"
            print(f"  [{done}/{total_combos}] {label}")
            try:
                metrics, cum = backtest(prices, log_ret, freq_label,
                                        freq_offset, opt, benchmark_log)
                metrics["Label"]    = label
                metrics["Strategy"] = strat_name
                metrics["Optimiser"]= opt
                metrics["Frequency"]= freq_label
                all_results.append(metrics)
                all_cumrets[label]  = cum
            except Exception as e:
                print(f"    !! Failed: {e}")

results_df = pd.DataFrame(all_results).sort_values("Sharpe", ascending=False)
results_df.to_csv(f"{OUT}/rebalancing_results_all.csv", index=False)
print(f"\n  All {len(results_df)} combinations evaluated.")
print(f"  Top 10 by Sharpe Ratio:")
cols_to_show = ["Label","Sharpe","Ann Return","Ann Vol","Max Drawdown",
                "Calmar","Alpha","Turnover (ann)","Tx Cost Drag"]
cols_to_show = [c for c in cols_to_show if c in results_df.columns]
print(results_df[cols_to_show].head(10).to_string(index=False))


# ── Per-optimiser best-frequency chart (2×2 grid for 4 optimisers) ───────────
fig, axes = plt.subplots(2, 2, figsize=(20, 12), sharey=False)
axes_flat  = axes.flatten()
bench_norm = benchmark_cum / benchmark_cum.iloc[0]

for ax, opt in zip(axes_flat, OPTIMISERS):
    subset = results_df[results_df["Optimiser"] == opt].sort_values("Sharpe", ascending=False)
    for _, row in subset.iterrows():
        lbl = row["Label"]
        if lbl in all_cumrets:
            cum = all_cumrets[lbl]
            ax.plot(cum.index, cum / cum.iloc[0],
                    label=f"{row['Frequency']} (Sh={row['Sharpe']:.2f})",
                    linewidth=1.3)
    ax.plot(bench_norm.index, bench_norm.values,
            color="black", linestyle=":", linewidth=1.5, label="Market EW")
    ax.set_title(f"{opt}\nRebalancing Frequency Comparison")
    ax.set_xlabel("Date"); ax.set_ylabel("Growth of $1")
    ax.legend(fontsize=6); ax.grid(True, linestyle="--", alpha=0.4)

plt.suptitle("Rebalancing Frequency vs Optimiser — S&P 500 (with Transaction Costs & Dynamic Risk Budgeting)", fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUT}/rebalancing_frequency_comparison.png", dpi=150, bbox_inches="tight")
plt.close()

# ── Sharpe heatmap: frequency vs optimiser (best strategy per cell) ───────────
pivot_heat = results_df.groupby(["Optimiser","Frequency"])["Sharpe"].max().unstack()
fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(pivot_heat, annot=True, fmt=".2f", cmap="RdYlGn",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Sharpe Ratio"})
ax.set_title("Best Sharpe Ratio — Optimiser × Rebalancing Frequency")
plt.tight_layout()
plt.savefig(f"{OUT}/sharpe_heatmap.png", dpi=150)
plt.close()
print("  Saved: rebalancing_frequency_comparison.png, sharpe_heatmap.png")

# ── Fig 6.7 — §6.2.1  Sharpe by frequency per optimiser (line chart) ─────────
_freq_order67 = ["Buy & Hold", "Yearly", "Monthly", "Quarterly", "Weekly"]
_opt_cols67   = {"Max Sharpe": "#E53935", "Min Variance": "#1E88E5",
                 "Black-Litterman": "#8E24AA", "Risk Parity": "#43A047"}
_opt_ls67     = {"Max Sharpe": "-", "Min Variance": "--",
                 "Black-Litterman": "-.", "Risk Parity": ":"}

_hs_results67 = results_df[results_df["Strategy"] == "High Sharpe"]

fig, _ax67 = plt.subplots(figsize=(11, 5))
for _opt67 in OPTIMISERS:
    _sub67 = _hs_results67[_hs_results67["Optimiser"] == _opt67].copy()
    _sub67["_fo"] = pd.Categorical(_sub67["Frequency"], categories=_freq_order67, ordered=True)
    _sub67 = _sub67.sort_values("_fo")
    _sharpes67 = _sub67.set_index("Frequency").reindex(_freq_order67)["Sharpe"]
    _ax67.plot(range(len(_freq_order67)), _sharpes67.values,
               marker="o", linewidth=2.2, markersize=8,
               color=_opt_cols67[_opt67], linestyle=_opt_ls67[_opt67], label=_opt67)
    if not _sharpes67.empty and not np.isnan(_sharpes67.values[-1]):
        _ax67.text(len(_freq_order67) - 0.95, _sharpes67.values[-1],
                   f" {_sharpes67.values[-1]:.2f}", va="center",
                   fontsize=8.5, color=_opt_cols67[_opt67], fontweight="bold")

_ax67.set_xticks(range(len(_freq_order67)))
_ax67.set_xticklabels(_freq_order67, fontsize=9.5)
_ax67.set_ylabel("Sharpe Ratio (net, 5 bps)", fontsize=10)
_ax67.set_xlabel("Rebalancing Frequency  →  (left = less trading, right = more trading)",
                 fontsize=9.5)
_ax67.set_title(
    "Figure 6.7: Sharpe Ratio by Rebalancing Frequency for Each Optimiser — High Sharpe Universe\n"
    "Max Sharpe benefits most from frequent rebalancing; other optimisers are nearly flat.",
    fontsize=10.5, fontweight="bold"
)
_ax67.set_xlim(-0.3, len(_freq_order67) - 0.5)
_ax67.legend(fontsize=9, loc="upper left")
_ax67.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(f"{OUT}/fig67_sharpe_by_frequency.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig67_sharpe_by_frequency.png")

# ── Fig 5.5 — §5.2–5.4  Optimiser comparison — HS universe, best frequency ───
_hs_best55 = (results_df[results_df["Strategy"] == "High Sharpe"]
              .sort_values("Sharpe", ascending=False)
              .groupby("Optimiser").first().reset_index())
_opt_names55  = _hs_best55["Optimiser"].tolist()
_opt_cols55   = ["#E53935", "#8E24AA", "#43A047", "#1E88E5"]

fig, _axes55 = plt.subplots(1, 4, figsize=(15, 5))
_mets55 = [("Ann Return", "Annualised Return", True),
           ("Sharpe",     "Sharpe Ratio",      False),
           ("Max Drawdown", "Max Drawdown (abs.)", True),
           ("Turnover (ann)", "Ann. Turnover (%)", False)]

for _ax55, (_col55, _title55, _is_pct55) in zip(_axes55, _mets55):
    _vals55 = _hs_best55[_col55].abs().tolist()
    _bars55 = _ax55.bar(range(len(_opt_names55)), _vals55,
                        color=_opt_cols55[:len(_opt_names55)],
                        width=0.55, edgecolor="white")
    _ax55.set_xticks(range(len(_opt_names55)))
    _ax55.set_xticklabels(
        [o.replace("Black-Litterman","B-L").replace("Min Variance","MV")
          .replace("Max Sharpe","MS").replace("Risk Parity","RP")
         for o in _opt_names55], fontsize=8.5)
    _ax55.set_title(_title55, fontsize=9.5, fontweight="bold")
    if _is_pct55:
        _ax55.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _p: f"{x:.0%}"))
    for _bar55, _v55 in zip(_bars55, _vals55):
        _lbl55 = f"{_v55:.1%}" if _is_pct55 else f"{_v55:.3f}" if _col55 == "Sharpe" else f"{_v55:.0f}%"
        _ax55.text(_bar55.get_x() + _bar55.get_width()/2,
                   _bar55.get_height() + _bar55.get_height() * 0.02,
                   _lbl55, ha="center", va="bottom", fontsize=7.5, fontweight="bold")
    _ax55.grid(True, linestyle="--", alpha=0.35)

fig.suptitle(
    "Figure 5.5: Optimiser Performance — High Sharpe Universe (best frequency per optimiser)\n"
    "MS = Max Sharpe  |  B-L = Black–Litterman  |  RP = Risk Parity  |  MV = Min Variance",
    fontsize=10, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(f"{OUT}/fig55_optimiser_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig55_optimiser_comparison.png")

# ── Fig 6.8 — §6.4  Full 60-strategy ranking waterfall ───────────────────────
_ranked68 = results_df.reset_index(drop=True)
_ranked68["Rank"] = _ranked68.index + 1
_tier_colors68 = {
    "High Sharpe":    "#1565C0",
    "PCA Cluster":    "#2E7D32",
    "Low Volatility": "#6A1B9A",
}
_bar_colors68 = [_tier_colors68.get(s, "#888")
                 for s in _ranked68["Strategy"]]

fig, _ax68 = plt.subplots(figsize=(16, 6))
_ax68.bar(_ranked68["Rank"], _ranked68["Sharpe"],
          color=_bar_colors68, edgecolor="white", linewidth=0.4, width=0.85, zorder=3)

_bsr68 = (annualised_return(benchmark_log) - RISK_FREE_ANNUAL) / annualised_vol(benchmark_log)
_ax68.axhline(_bsr68, color="#E53935", linewidth=1.3, linestyle="--", zorder=4,
              label=f"EW Benchmark (Sh={_bsr68:.2f})")
_ax68.axhline(0, color="#888", linewidth=0.8, zorder=4)

# Tier dividers
for _div_x68 in [20.5, 40.5]:
    _ax68.axvline(_div_x68, color="#333", linewidth=1.5, linestyle=":", alpha=0.7, zorder=5)

# Tier labels
_ymax68 = _ranked68["Sharpe"].max()
for _x0, _x1, _lbl68, _col68 in [
    (1,  20, "Tier 1 — High Sharpe\n(Ranks 1–20)",     "#1565C0"),
    (21, 40, "Tier 2 — PCA Cluster\n(Ranks 21–40)",    "#2E7D32"),
    (41, 60, "Tier 3 — Low Volatility\n(Ranks 41–60)", "#6A1B9A"),
]:
    _ax68.text((_x0+_x1)/2, _ymax68 * 1.07, _lbl68, ha="center",
               fontsize=8.5, color=_col68, fontweight="bold",
               bbox=dict(fc=_col68, ec="none", alpha=0.12, pad=3, boxstyle="round,pad=0.4"))

_ax68.set_xlabel("Strategy Rank", fontsize=10)
_ax68.set_ylabel("Sharpe Ratio (net of 5 bps costs)", fontsize=10)
_ax68.set_title(
    "Figure 6.8: Full 60-Strategy Ranking by Sharpe Ratio — Three-Tier Structure\n"
    "Universe choice creates two sharp discontinuities; within-tier spread reflects optimiser and frequency choices.",
    fontsize=11, fontweight="bold"
)
import matplotlib.patches as _mpatches
_leg_handles68 = [
    _mpatches.Patch(color="#1565C0", label="High Sharpe"),
    _mpatches.Patch(color="#2E7D32", label="PCA Cluster"),
    _mpatches.Patch(color="#6A1B9A", label="Low Volatility"),
    plt.Line2D([0], [0], linestyle="--", color="#E53935",
               label=f"EW Benchmark (Sh={_bsr68:.2f})"),
]
_ax68.legend(handles=_leg_handles68, fontsize=9, loc="upper right")
_ax68.set_xlim(0.4, 60.6)
_ax68.set_xticks([1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60])
_ax68.grid(True, linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig(f"{OUT}/fig68_strategy_ranking_waterfall.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig68_strategy_ranking_waterfall.png")

# ── Fig 6.4 — §6.2  Turnover vs Sharpe scatter (HS universe) ─────────────────
_hs_to64 = results_df[results_df["Strategy"] == "High Sharpe"].copy()
_opt_cols64 = {"Max Sharpe": "#E53935", "Min Variance": "#1E88E5",
               "Black-Litterman": "#8E24AA", "Risk Parity": "#43A047"}
_freq_markers64 = {"Weekly": "o", "Monthly": "s", "Quarterly": "^",
                   "Yearly": "D", "Buy & Hold": "P"}

fig, _ax64 = plt.subplots(figsize=(11, 7))
for _, _row64 in _hs_to64.iterrows():
    _opt64  = _row64["Optimiser"]
    _freq64 = _row64["Frequency"]
    _ax64.scatter(
        _row64.get("Turnover (ann)", 0) * 100,
        _row64["Sharpe"],
        color=_opt_cols64.get(_opt64, "#555"),
        marker=_freq_markers64.get(_freq64, "o"),
        s=110, zorder=4, edgecolors="white", linewidths=0.7,
        label=f"{_opt64} · {_freq64}",
    )

# Legend — optimiser colour patches
import matplotlib.patches as _mpatches64
_opt_patches64 = [_mpatches64.Patch(color=c, label=o)
                  for o, c in _opt_cols64.items()]
_freq_handles64 = [plt.scatter([], [], marker=m, color="grey", s=80, label=f)
                   for f, m in _freq_markers64.items()]
_leg1_64 = _ax64.legend(handles=_opt_patches64, title="Optimiser",
                         loc="lower right", fontsize=8.5, framealpha=0.9)
_ax64.add_artist(_leg1_64)
_ax64.legend(handles=_freq_handles64, title="Frequency",
             loc="upper right", fontsize=8.5, framealpha=0.9)

_ax64.set_xlabel("Annualised One-Way Turnover (%)", fontsize=10)
_ax64.set_ylabel("Sharpe Ratio (net of 5 bps)", fontsize=10)
_ax64.set_title(
    f"Figure 6.4: Turnover vs Sharpe Trade-off — High Sharpe Universe "
    f"(cost: {TRANSACTION_COST_BPS} bps)\nAll 4 optimisers × 5 rebalancing frequencies",
    fontsize=11, fontweight="bold"
)
_ax64.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(f"{OUT}/fig64_turnover_sharpe_tradeoff.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig64_turnover_sharpe_tradeoff.png")

# ── Fig 6.5 — §6.4  Calmar ratio heatmap ─────────────────────────────────────
_hs_calmar65 = results_df[results_df["Strategy"] == "High Sharpe"].copy()
_pivot_calmar65 = _hs_calmar65.groupby(["Optimiser","Frequency"])["Calmar"].max().unstack()
_freq_order65 = [f for f in ["Weekly","Monthly","Quarterly","Yearly","Buy & Hold"]
                 if f in _pivot_calmar65.columns]
_pivot_calmar65 = _pivot_calmar65[_freq_order65]

fig, _ax65 = plt.subplots(figsize=(11, 5))
sns.heatmap(_pivot_calmar65, annot=True, fmt=".2f", cmap="YlGn",
            linewidths=0.6, ax=_ax65, annot_kws={"size": 10},
            cbar_kws={"label": "Calmar Ratio", "shrink": 0.75})
_ax65.set_title(
    "Figure 6.5: Calmar Ratio (Ann. Return / |Max Drawdown|) — High Sharpe Universe\n"
    "Higher = better drawdown-adjusted return. Net of 5 bps costs.",
    fontsize=11, fontweight="bold"
)
_ax65.set_xlabel("Rebalancing Frequency", fontsize=10)
_ax65.set_ylabel("Optimiser", fontsize=10)
_ax65.tick_params(axis="x", rotation=0)
_ax65.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.savefig(f"{OUT}/fig65_calmar_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig65_calmar_heatmap.png")

# ── Fig 3.4 — §3.6  Rolling 30-day volatility regimes ────────────────────────
_rolling_vol34 = benchmark_log.rolling(30).std() * np.sqrt(252)

fig, _ax34 = plt.subplots(figsize=(14, 5))
_ax34.plot(benchmark_log.index, _rolling_vol34.values,
           color="#1A237E", linewidth=1.4, zorder=3, label="30-day Rolling Vol")
_ax34.axhline(0.25, color="#E53935", linestyle=":", linewidth=1.2, alpha=0.8,
              label="25% dynamic risk-cap trigger")
_ax34.fill_between(benchmark_log.index, 0, _rolling_vol34.values, alpha=0.12, color="#1A237E")
_ax34.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _p: f"{x:.0%}"))
_ax34.set_ylabel("30-day Rolling Volatility (annualised)", fontsize=10)
_ax34.set_xlabel("Date", fontsize=10)
_ax34.set_title(
    "Figure 3.4: Rolling 30-day Realised Volatility — Equal-Weighted S&P 500 Benchmark, 2020–2026\n"
    "Dashed line = 25% threshold at which dynamic risk budgeting tightens per-asset weight cap.",
    fontsize=11, fontweight="bold"
)
_ax34.legend(fontsize=9)
_ax34.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(f"{OUT}/fig34_volatility_regimes.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig34_volatility_regimes.png")

# ── Fig 5.3 — §5.5  Sharpe distribution boxplot by universe ──────────────────
fig, _ax53 = plt.subplots(figsize=(10, 6))
_univ_order53 = ["High Sharpe", "PCA Cluster", "Low Volatility"]
_univ_cols53  = {"High Sharpe": "#1565C0", "PCA Cluster": "#2E7D32", "Low Volatility": "#6A1B9A"}

_bp_data53 = [results_df[results_df["Strategy"] == u]["Sharpe"].dropna().values
              for u in _univ_order53]
_bp53 = _ax53.boxplot(_bp_data53, patch_artist=True, widths=0.45,
                      medianprops=dict(color="white", linewidth=2),
                      whiskerprops=dict(linewidth=1.4),
                      capprops=dict(linewidth=1.4),
                      flierprops=dict(marker="o", markersize=5, alpha=0.6))
for _patch53, _uname53 in zip(_bp53["boxes"], _univ_order53):
    _patch53.set_facecolor(_univ_cols53[_uname53])
    _patch53.set_alpha(0.75)

# Overlay strip
for _i53, (_uname53, _vals53) in enumerate(zip(_univ_order53, _bp_data53), start=1):
    _jitter53 = np.random.default_rng(99).uniform(-0.12, 0.12, len(_vals53))
    _ax53.scatter(_i53 + _jitter53, _vals53, color=_univ_cols53[_uname53],
                  s=30, alpha=0.45, zorder=4)

_ax53.axhline(_bsr68, color="#E53935", linewidth=1.3, linestyle="--",
              label=f"EW Benchmark (Sh={_bsr68:.2f})")
_ax53.set_xticks(range(1, len(_univ_order53)+1))
_ax53.set_xticklabels(_univ_order53, fontsize=10)
_ax53.set_ylabel("Sharpe Ratio (net of 5 bps costs)", fontsize=10)
_ax53.set_title(
    "Figure 5.3: Distribution of Sharpe Ratios Across All 60 Strategies, Grouped by Universe\n"
    "Each point = one strategy (4 optimisers × 5 frequencies). Dashed = EW benchmark.",
    fontsize=11, fontweight="bold"
)
_ax53.legend(fontsize=9)
_ax53.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(f"{OUT}/fig53_universe_sharpe_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: fig53_universe_sharpe_distribution.png")



STAGE 7 — REBALANCING BACKTESTER
  [1/60] High Sharpe | Max Sharpe | Weekly
  [2/60] High Sharpe | Max Sharpe | Monthly
  [3/60] High Sharpe | Max Sharpe | Quarterly
  [4/60] High Sharpe | Max Sharpe | Yearly
  [5/60] High Sharpe | Max Sharpe | Buy & Hold
  [6/60] High Sharpe | Min Variance | Weekly
  [7/60] High Sharpe | Min Variance | Monthly
  [8/60] High Sharpe | Min Variance | Quarterly
  [9/60] High Sharpe | Min Variance | Yearly
  [10/60] High Sharpe | Min Variance | Buy & Hold
  [11/60] High Sharpe | Black-Litterman | Weekly
  [12/60] High Sharpe | Black-Litterman | Monthly
  [13/60] High Sharpe | Black-Litterman | Quarterly
  [14/60] High Sharpe | Black-Litterman | Yearly
  [15/60] High Sharpe | Black-Litterman | Buy & Hold
  [16/60] High Sharpe | Risk Parity | Weekly
  [17/60] High Sharpe | Risk Parity | Monthly
  [18/60] High Sharpe | Risk Parity | Quarterly
  [19/60] High Sharpe | Risk Parity | Yearly
  [20/60] High Sharpe | Risk Parity | Buy & Hold
  [21/60] Low Volatilit

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STAGE 8 — EFFICIENT FRONTIER (Max Sharpe + Min Var + BL all on one plot)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n" + "="*70)
print("STAGE 8 — EFFICIENT FRONTIER")
print("="*70)

ef_points = []   # {label, vol, ret, sharpe}

# Simulate the full frontier curve using random portfolios (Monte Carlo)
def monte_carlo_frontier(prices, n_portfolios=3000):
    log_r = np.log(prices / prices.shift(1)).dropna()
    mu    = log_r.mean() * 252
    cov   = log_r.cov() * 252
    n     = len(prices.columns)
    vols, rets, srs = [], [], []
    for _ in range(n_portfolios):
        w   = np.random.dirichlet(np.ones(n))
        r   = w.dot(mu)
        v   = np.sqrt(w @ cov.values @ w)
        sr  = (r - RISK_FREE_ANNUAL) / v if v > 0 else 0
        vols.append(v); rets.append(r); srs.append(sr)
    return np.array(vols), np.array(rets), np.array(srs)

# Use the High Sharpe universe for the frontier (largest return spread)
hs_prices = pivot_df[top_sharpe].dropna(axis=1, how="any")

print("  Simulating Monte Carlo frontier (3000 portfolios)...")
mc_vols, mc_rets, mc_srs = monte_carlo_frontier(hs_prices)

# Get exact optimal points for each optimiser
opt_points = {}
for opt in OPTIMISERS:
    w = get_weights(hs_prices, opt)
    w_arr = np.array([w.get(t, 0) for t in hs_prices.columns])
    if w_arr.sum() > 0: w_arr /= w_arr.sum()
    log_r = np.log(hs_prices / hs_prices.shift(1)).dropna()
    ret   = log_r.dot(w_arr)
    ar    = annualised_return(ret)
    av    = annualised_vol(ret)
    sr    = (ar - RISK_FREE_ANNUAL) / av if av > 0 else 0
    opt_points[opt] = {"vol": av, "ret": ar, "sharpe": sr, "weights": w}
    ef_points.append({"Optimiser": opt, "Volatility": av,
                      "Annual Return": ar, "Sharpe": sr})
    print(f"  {opt:20s} | Vol={av:.2%}  Ret={ar:.2%}  Sharpe={sr:.2f}")

# Plot
fig, ax = plt.subplots(figsize=(13, 8))

sc = ax.scatter(mc_vols, mc_rets, c=mc_srs, cmap="RdYlGn",
                s=15, alpha=0.5, zorder=1)
plt.colorbar(sc, ax=ax, label="Sharpe Ratio")

markers = {"Max Sharpe": ("*", "#e74c3c", 350),
           "Min Variance": ("D", "#3498db", 180),
           "Black-Litterman": ("^", "#9b59b6", 220),
           "Risk Parity": ("o", "#43A047", 250)}

for opt, pt in opt_points.items():
    mk, col, sz = markers[opt]
    ax.scatter(pt["vol"], pt["ret"], marker=mk, color=col, s=sz,
               zorder=5, edgecolors="black", linewidths=0.8,
               label=f"{opt}  (Sh={pt['sharpe']:.2f})")
    ax.annotate(opt,
                (pt["vol"], pt["ret"]),
                textcoords="offset points", xytext=(10, 5),
                fontsize=9, fontweight="bold", color=col)

# Market benchmark dot
bv = annualised_vol(benchmark_log)
br = annualised_return(benchmark_log)
ax.scatter(bv, br, marker="X", color="black", s=250, zorder=5,
           edgecolors="white", linewidths=0.8, label=f"Market EW  (Sh={(br-RISK_FREE_ANNUAL)/bv:.2f})")

ax.set_xlabel("Annualised Volatility", fontsize=11)
ax.set_ylabel("Annualised Return", fontsize=11)
ax.set_title("Efficient Frontier — Max Sharpe · Min Variance · Black-Litterman\n"
             f"Universe: High Sharpe tickers ({', '.join(top_sharpe)})", fontsize=12)
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(f"{OUT}/efficient_frontier_all_optimisers.png", dpi=160, bbox_inches="tight")
plt.close()

# Save frontier table
pd.DataFrame(ef_points).to_csv(f"{OUT}/efficient_frontier_points.csv", index=False)
print("  Saved: efficient_frontier_all_optimisers.png")

In [20]:

# ══════════════════════════════════════════════════════════════════════════════
# STAGE 9 — STRATEGY RECOMMENDATION ENGINE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("STAGE 9 — STRATEGY RECOMMENDATION ENGINE")
print("="*70)

# Rank by Sharpe Ratio — primary criterion
ranked = results_df.sort_values("Sharpe", ascending=False).reset_index(drop=True)
winner = ranked.iloc[0]

# Runner-up for comparison
runner_up = ranked.iloc[1]

# Best per optimiser
best_per_opt = ranked.groupby("Optimiser").first().reset_index()

# Best per frequency
best_per_freq = ranked.groupby("Frequency").first().reset_index()

# ── Written recommendation ────────────────────────────────────────────────────
def format_pct(x): return f"{x:.2%}"
def format_f(x):   return f"{x:.4f}"

recommendation = f"""
================================================================================
STRATEGY RECOMMENDATION REPORT — S&P 500 Portfolio Analysis
================================================================================
Selection Criterion : Sharpe Ratio (annualised return per unit of risk)
Transaction Costs   : {TRANSACTION_COST_BPS} bps per side (realistic liquid-ETF estimate)
Dynamic Risk Budget : {"Enabled" if DYNAMIC_RISK_BUDGET else "Disabled"} (vol-scaled weight cap = {VOL_WEIGHT_CAP:.0%})
Data Period         : {log_returns_df.index.min().date()} → {log_returns_df.index.max().date()}
Tickers Analysed    : {log_returns_df.shape[1]} S&P 500 constituents
Combinations Tested : {len(results_df)} (3 strategies × 4 optimisers × 5 frequencies)
Best ML Forecaster  : {best_ml_model}

────────────────────────────────────────────────────────────────────────────────
★  RECOMMENDED STRATEGY
────────────────────────────────────────────────────────────────────────────────
Label             : {winner['Label']}
Strategy Universe : {winner['Strategy']}
Optimiser         : {winner['Optimiser']}
Rebal Frequency   : {winner['Frequency']}

Performance Metrics:
  Sharpe Ratio       : {format_f(winner['Sharpe'])}
  Annualised Return  : {format_pct(winner['Ann Return'])}
  Annualised Vol     : {format_pct(winner['Ann Vol'])}
  Max Drawdown       : {format_pct(winner['Max Drawdown'])}
  Calmar Ratio       : {format_f(winner['Calmar'])}
  Cumulative Return  : {format_pct(winner['Cumulative Return'])}
  Alpha (vs EW mkt)  : {format_pct(winner.get('Alpha', float('nan')))}

WHY THIS STRATEGY WAS SELECTED:
  Out of {len(results_df)} tested combinations, this strategy achieved the highest
  Sharpe Ratio of {winner['Sharpe']:.4f}, meaning it delivered the greatest
  risk-adjusted return over the 2020–2024 period. It uses the
  '{winner['Strategy']}' ticker universe — stocks screened for
  {'strong historical risk-adjusted momentum' if winner['Strategy']=='High Sharpe'
   else 'low historical volatility' if winner['Strategy']=='Low Volatility'
   else 'structural diversification via PCA clustering'} —
  combined with {winner['Optimiser']} portfolio construction
  ({'maximising the Sharpe ratio of the portfolio directly via mean-variance optimisation'
    if winner['Optimiser']=='Max Sharpe'
    else 'minimising overall portfolio variance regardless of expected return'
    if winner['Optimiser']=='Min Variance'
    else 'incorporating market-implied returns blended with an analyst view of 10% return'}).
  Rebalancing {winner['Frequency']} provided the optimal trade-off between
  {'capturing fast momentum' if winner['Frequency']=='Weekly'
   else 'tracking medium-term trends while limiting transaction costs'
   if winner['Frequency'] in ['Monthly','Quarterly']
   else 'strategic long-term positioning with minimal turnover'
   if winner['Frequency']=='Yearly'
   else 'zero transaction cost drag by never rebalancing'}.

────────────────────────────────────────────────────────────────────────────────
2nd PLACE
────────────────────────────────────────────────────────────────────────────────
  {runner_up['Label']}
  Sharpe={runner_up['Sharpe']:.4f}  |  Return={runner_up['Ann Return']:.2%}
  |  Vol={runner_up['Ann Vol']:.2%}  |  MDD={runner_up['Max Drawdown']:.2%}

The winner outperformed the runner-up by {winner['Sharpe']-runner_up['Sharpe']:.4f}
Sharpe points ({(winner['Sharpe']/runner_up['Sharpe']-1)*100:.1f}% better risk-adjusted return).

────────────────────────────────────────────────────────────────────────────────
BEST PER OPTIMISER
────────────────────────────────────────────────────────────────────────────────"""

for _, row in best_per_opt.iterrows():
    recommendation += f"""
  {row['Optimiser']:20s} → {row['Frequency']:12s} | Sharpe={row['Sharpe']:.4f}  Return={row['Ann Return']:.2%}  MDD={row['Max Drawdown']:.2%}"""

recommendation += f"""

────────────────────────────────────────────────────────────────────────────────
BEST PER REBALANCING FREQUENCY
────────────────────────────────────────────────────────────────────────────────"""

for _, row in best_per_freq.iterrows():
    recommendation += f"""
  {row['Frequency']:15s} → {row['Optimiser']:20s} | Sharpe={row['Sharpe']:.4f}  Return={row['Ann Return']:.2%}  MDD={row['Max Drawdown']:.2%}"""

recommendation += f"""

────────────────────────────────────────────────────────────────────────────────
EFFICIENT FRONTIER SUMMARY (High Sharpe Universe)
────────────────────────────────────────────────────────────────────────────────"""

for opt, pt in opt_points.items():
    recommendation += f"""
  {opt:20s} | Vol={pt['vol']:.2%}  Return={pt['ret']:.2%}  Sharpe={pt['sharpe']:.4f}"""

recommendation += f"""

────────────────────────────────────────────────────────────────────────────────
ML FORECAST MODEL RANKING (by RMSE on held-out test set)
────────────────────────────────────────────────────────────────────────────────"""

for _, row in ml_leaderboard.iterrows():
    star = "★ BEST" if row["Model"] == best_ml_model else "      "
    recommendation += f"""
  {star} {row['Model']:20s} | RMSE={row['Avg_RMSE']:.6f}  MAE={row['Avg_MAE']:.6f}  Dir Acc={row['Avg_Dir_Acc']:.2%}"""

recommendation += f"""

  The best ML model ({best_ml_model}) achieved the lowest RMSE and is
  recommended for use in the forecasting/rebalancing signal layer.

================================================================================
END OF REPORT
================================================================================
"""

print(recommendation)

with open(f"{OUT}/strategy_recommendation.txt", "w") as f:
    f.write(recommendation)
print("  Saved: strategy_recommendation.txt")



STAGE 9 — STRATEGY RECOMMENDATION ENGINE

STRATEGY RECOMMENDATION REPORT — S&P 500 Portfolio Analysis
Selection Criterion : Sharpe Ratio (annualised return per unit of risk)
Transaction Costs   : 5 bps per side (realistic liquid-ETF estimate)
Dynamic Risk Budget : Enabled (vol-scaled weight cap = 30%)
Data Period         : 2020-01-03 → 2026-05-15
Tickers Analysed    : 488 S&P 500 constituents
Combinations Tested : 60 (3 strategies × 4 optimisers × 5 frequencies)
Best ML Forecaster  : ARIMA

────────────────────────────────────────────────────────────────────────────────
★  RECOMMENDED STRATEGY
────────────────────────────────────────────────────────────────────────────────
Label             : High Sharpe | Max Sharpe | Weekly
Strategy Universe : High Sharpe
Optimiser         : Max Sharpe
Rebal Frequency   : Weekly

Performance Metrics:
  Sharpe Ratio       : 1.7099
  Annualised Return  : 52.41%
  Annualised Vol     : 27.58%
  Max Drawdown       : -41.48%
  Calmar Ratio       : 1.2637


In [21]:

# ══════════════════════════════════════════════════════════════════════════════
# STAGE 10 — FINAL SUMMARY CHART & OUTPUT MANIFEST
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("STAGE 10 — FINAL SUMMARY CHART")
print("="*70)

# Top-10 strategies on one cumulative return chart
fig, ax = plt.subplots(figsize=(16, 8))
top10 = ranked.head(10)
cmap  = plt.cm.tab10(np.linspace(0, 1, 10))

for i, (_, row) in enumerate(top10.iterrows()):
    lbl = row["Label"]
    if lbl in all_cumrets:
        cum = all_cumrets[lbl]
        ax.plot(cum.index, cum / cum.iloc[0], color=cmap[i], linewidth=1.5,
                label=f"#{i+1} {row['Frequency']} | {row['Optimiser']} (Sh={row['Sharpe']:.2f})")

ax.plot(bench_norm.index, bench_norm.values,
        color="black", linestyle=":", linewidth=2, label="Market EW Benchmark")
ax.set_title("Top 10 Strategies by Sharpe Ratio — S&P 500 (2020–2024)", fontsize=13)
ax.set_xlabel("Date"); ax.set_ylabel("Growth of $1")
ax.legend(fontsize=7.5, loc="upper left")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(f"{OUT}/top10_strategies_cumulative.png", dpi=150, bbox_inches="tight")
plt.close()

# Full ranked table saved
ranked.to_csv(f"{OUT}/all_strategies_ranked.csv", index=False)
print("  Saved: top10_strategies_cumulative.png, all_strategies_ranked.csv")

# ── Output manifest ───────────────────────────────────────────────────────────
print("\n" + "="*70)
print("PIPELINE COMPLETE — Output files")
print("="*70)
for f in sorted(os.listdir(OUT)):
    sz = os.path.getsize(os.path.join(OUT, f))
    print(f"  {f:<50} {sz/1024:>8.1f} KB")



STAGE 10 — FINAL SUMMARY CHART
  Saved: top10_strategies_cumulative.png, all_strategies_ranked.csv

PIPELINE COMPLETE — Output files
  all_strategies_ranked.csv                              15.8 KB
  correlation_clustermap.png                            180.0 KB
  descriptive_stats.csv                                  40.8 KB
  efficient_frontier_all_optimisers.png                 341.6 KB
  efficient_frontier_points.csv                           0.3 KB
  fig34_volatility_regimes.png                          140.0 KB
  fig35_benchmark_wealth_rolling_sharpe.png             162.3 KB
  fig53_universe_sharpe_distribution.png                 98.1 KB
  fig54_universe_profiles.png                           134.4 KB
  fig55_optimiser_comparison.png                        124.3 KB
  fig56_universe_wealth_curves.png                      160.9 KB
  fig64_turnover_sharpe_tradeoff.png                     92.6 KB
  fig65_calmar_heatmap.png                               83.8 KB
  fig66_bias_variance

In [25]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Check price data
print("Price CSV:", os.path.exists("/content/drive/MyDrive/Klurvss3am/from 2020_all_sp500_tickers_since_2020.csv"))

# Check pipeline outputs
out_folder = "/content/drive/MyDrive/Klurvss3am/latest"
print("\nCSVs in output folder:")
for f in sorted(os.listdir(out_folder)):
    if f.endswith(".csv"):
        print(f"  ✅ {f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Price CSV: True

CSVs in output folder:
  ✅ all_strategies_ranked.csv
  ✅ descriptive_stats.csv
  ✅ efficient_frontier_points.csv
  ✅ from 2020_all_sp500_tickers_since_2020.csv
  ✅ ml_model_detail.csv
  ✅ ml_model_leaderboard.csv
  ✅ rebalancing_results_all.csv


In [14]:
# Read dashboard
with open("portfolio_dashboard.py", "r") as f:
    src = f.read()

# Replace the two path lines to point at your Drive
src = src.replace(
    'PIPELINE_OUT  = os.environ.get("PIPELINE_OUT", os.path.dirname(os.path.abspath(__file__)))',
    'PIPELINE_OUT  = "/content/drive/MyDrive/Klurvss3am/latest"'
)
src = src.replace(
    'DATA_CSV      = os.path.join(PIPELINE_OUT, "from_2020_all_sp500_tickers_since_2020.csv")',
    'DATA_CSV      = "/content/drive/MyDrive/Klurvss3am/from 2020_all_sp500_tickers_since_2020.csv"'
)

with open("portfolio_dashboard.py", "w") as f:
    f.write(src)

print("✅ Paths patched")

✅ Paths patched


In [15]:
# Fix 1: correct package name for pypfopt
!pip install pyportfolioopt -q

# Fix 2: install pyngrok properly
!pip install pyngrok -q

# Fix 3: install ngrok binary
!wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xzf ngrok-v3-stable-linux-amd64.tgz
!mv ngrok /usr/local/bin/ngrok
!pip install streamlit pyportfolioopt pyngrok plotly scipy scikit-learn -q
# Verify
!ngrok --version
print("✅ All installed")

ngrok version 3.39.5
✅ All installed


In [23]:
# Set ngrok token (get free one from ngrok.com)
!ngrok authtoken 3EUoLNYQ54jAmvzRtcJvoeimawc_3TyX5zH6rRqA5ccKFjzhg

import subprocess, threading, time
from pyngrok import ngrok

!pkill -f streamlit 2>/dev/null
time.sleep(2)

def run_streamlit():
    subprocess.run([
        "streamlit", "run", "/content/portfolio_dashboard.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ])

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(8)

url = ngrok.connect(8501)
print(f"\n✅ Dashboard live at:\n{url}")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
^C


PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: Your account may not run more than 5 endpoints over a single ngrok agent session.\nThe endpoints already running on this session are:\ntn_3EVwJ0Tspb5RpzacdKKVfmOMkrj, tn_3EVqsOzM4PP4LznorLkdPgMtZq4, tn_3EVr3fXVC4WUB274K2hE1EjBmPJ, tn_3EVvFI4szkt0nsogzKuq6T7XkDk, tn_3EVvVXaA3SEW1pl51MzTJSF0dro.\nUpgrade to a Pay-as-you-go plan at: https://dashboard.ngrok.com/billing/choose-a-plan?plan=paygo\r\n\r\nERR_NGROK_324\r\n"}}
